# 01 - Reproduction Experiments

This notebook is the main reproduction notebook for the project.

Scope:
- Table 1: main comparison on CIFAR-10, CIFAR-100-20, STL-10, ImageNet-10
- Table 2: convergence of inferred K
- Table 3: ImageNet-50 paper summary and local status
- Deviation analysis for the reproduction section


## Optional Regeneration

Current local PnP preset in this repo:
- `epochs=80`
- `warmup_epochs=30`
- `lambda_param=2.0`

The next code cell refreshes the structured Table 1 view whenever the source CSV is newer so the notebook does not keep stale numbers.


In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

ROOT = Path('..').resolve()
RESULT_DIR = ROOT / 'data' / 'scan_results'
RESULT_DIR.mkdir(parents=True, exist_ok=True)

required_datasets = ['cifar-10', 'cifar-20', 'stl-10', 'imagenet-10']
raw_dataset_checks = {
    'cifar-10': [ROOT / 'data' / 'cifar-10-batches-py'],
    'cifar-20': [ROOT / 'data' / 'cifar-100-python'],
    'stl-10': [ROOT / 'data' / 'stl10_binary'],
    'imagenet-10': [ROOT / 'data' / 'scan_cache' / 'imagenet-10_train_features.pt'],
}

cache_checks = {
    dataset: [
        ROOT / 'data' / 'scan_cache' / f'{dataset}_train_features.pt',
        ROOT / 'data' / 'scan_cache' / f'{dataset}_eval_features.pt',
    ]
    for dataset in required_datasets
}

missing_datasets = []
for dataset in required_datasets:
    raw_ready = all(path.exists() for path in raw_dataset_checks[dataset])
    cache_ready = all(path.exists() for path in cache_checks[dataset])
    print(f'[check] {dataset}: raw_ready={raw_ready} cache_ready={cache_ready}')
    if not (raw_ready and cache_ready):
        missing_datasets.append(dataset)

if missing_datasets:
    print(f'[prepare] missing datasets or caches detected: {missing_datasets}')
    subprocess.run(
        [sys.executable, 'scripts/prepare_and_run_report.py', '--datasets', *missing_datasets],
        check=True,
        cwd=ROOT,
    )
else:
    print('[prepare] all required datasets and caches are already available.')

required_tables = [
    RESULT_DIR / 'table1_main_comparison.csv',
    RESULT_DIR / 'table1_paper_vs_local.csv',
    RESULT_DIR / 'table2_inferred_k.csv',
    RESULT_DIR / 'table3_imagenet50_summary.csv',
]

if any(not path.exists() for path in required_tables):
    subprocess.run(
        [sys.executable, 'scripts/generate_reproduction_tables.py', '--tables', 'table1', 'table2', 'table3'],
        check=True,
        cwd=ROOT,
    )

main_table_path = RESULT_DIR / 'table1_main_comparison.csv'
structured_path = RESULT_DIR / 'table1_structured_for_report.csv'
if (not structured_path.exists()) or structured_path.stat().st_mtime < main_table_path.stat().st_mtime:
    subprocess.run([sys.executable, 'scripts/format_table1.py'], check=True, cwd=ROOT)

table1 = pd.read_csv(structured_path)
comparison_table = pd.read_csv(RESULT_DIR / 'table1_paper_vs_local.csv')
table2 = pd.read_csv(RESULT_DIR / 'table2_inferred_k.csv')
table3 = pd.read_csv(RESULT_DIR / 'table3_imagenet50_summary.csv')

RESULT_DIR


[check] cifar-10: raw_ready=True cache_ready=True
[check] cifar-20: raw_ready=True cache_ready=True
[check] stl-10: raw_ready=True cache_ready=True
[check] imagenet-10: raw_ready=True cache_ready=True
[prepare] all required datasets and caches are already available.


WindowsPath('D:/Coding/Data mining/Group4_Lab3/Group4_Lab3/data/scan_results')

## Table 1

Structured view for the main comparison section of the report. This table is now synchronized with `table1_main_comparison.csv`.


In [2]:
cols = table1.columns.tolist()
multi = []
for c in cols:
    if c == 'Method':
        multi.append(('Method', 'Metric'))
    else:
        ds, metric = c.split('|')
        multi.append((ds, metric))

table1_multi = table1.copy()
table1_multi.columns = pd.MultiIndex.from_tuples(multi)
display(table1_multi)


Method CIFAR-10             CIFAR-100             STL-10              \
         Metric      NMI   ACC   ARI       NMI   ACC   ARI    NMI   ACC   ARI   
0  SCAN (paper)     71.2  81.8  66.5      44.1  42.2  26.7   65.4  75.5  59.0   
1  SCAN (local)     65.4  74.9  58.8      42.3  41.4  25.9   63.8  73.5  57.1   
2    PnP (K0=3)     71.9  82.4  67.5      45.2  43.8  28.1   64.3  74.5  57.6   
3   PnP (K0=20)     71.1  81.6  66.2         -     -     -   65.1  74.7  58.9   
4   PnP (K0=30)        -     -     -      44.9  43.1  27.8      -     -     -   
5   Ours (K0=3)     54.7  55.9  43.6      31.5  27.5  16.3   41.8  28.8  24.6   
6  Ours (K0=20)     62.5  71.7  55.8         -     -     -   53.6  57.7  41.7   
7  Ours (K0=30)        -     -     -      41.7  41.0  24.9      -     -     -   

  ImageNet-10              
          NMI   ACC   ARI  
0        86.2  92.0  83.3  
1        92.6  96.8  93.0  
2        88.6  91.2  87.1  
3        86.9  91.8  84.7  
4           -     -     -  
5        55.3  29.8  31.8  
6        87.5  88.8  84.0  
7           -     -     -

## Deviation Analysis

The reproduction requirement asks for explicit paper-vs-local comparison together with relative deviation. The next cells expose that comparison directly instead of relying only on the formatted report table.


In [3]:
mean_deviation = (
    comparison_table.groupby(['Dataset', 'Paper method'])['Relative deviation(%)']
    .mean()
    .reset_index()
    .sort_values('Relative deviation(%)', ascending=False)
)
display(mean_deviation)

worst_rows = comparison_table.sort_values('Relative deviation(%)', ascending=False).head(12)
display(worst_rows)


,Dataset,Paper method,Relative deviation(%)
7,imagenet-10,"PnP (paper, K0=3)",56.146174
10,stl-10,"PnP (paper, K0=3)",51.195086
3,cifar-20,"PnP (paper, K0=3)",36.523535
1,cifar-10,"PnP (paper, K0=3)",30.502865
9,stl-10,"PnP (paper, K0=20)",23.203306
0,cifar-10,"PnP (paper, K0=20)",13.298582
2,cifar-10,SCAN,9.353424
8,imagenet-10,SCAN,8.087875
4,cifar-20,"PnP (paper, K0=30)",7.396334
5,cifar-20,SCAN,2.947015


,Dataset,Paper method,Local method,Metric,Paper result,Our result,Relative deviation(%)
30,imagenet-10,"PnP (paper, K0=3)",ours,ACC(%),91.2,29.807692,67.316127
32,imagenet-10,"PnP (paper, K0=3)",ours,ARI(%),87.1,31.758227,63.538201
21,stl-10,"PnP (paper, K0=3)",ours,ACC(%),74.5,28.750000,61.409396
23,stl-10,"PnP (paper, K0=3)",ours,ARI(%),57.6,24.629388,57.240646
14,cifar-20,"PnP (paper, K0=3)",ours,ARI(%),28.1,16.286555,42.040729
31,imagenet-10,"PnP (paper, K0=3)",ours,NMI(%),88.6,55.300403,37.584195
12,cifar-20,"PnP (paper, K0=3)",ours,ACC(%),43.8,27.490000,37.237443
5,cifar-10,"PnP (paper, K0=3)",ours,ARI(%),67.5,43.585752,35.428516
22,stl-10,"PnP (paper, K0=3)",ours,NMI(%),64.3,41.836656,34.935216
3,cifar-10,"PnP (paper, K0=3)",ours,ACC(%),82.4,55.860000,32.208738


## Interpretation

Main takeaways for the report text:
- `SCAN (local)` is reasonably close to the paper on `cifar-20` and `stl-10`, and even exceeds the paper on `imagenet-10`.
- The largest gaps appear in `Ours (K0=3)`, especially on `stl-10` and `imagenet-10`. This suggests the local split and merge schedule is still sensitive when the initial cluster count starts far below the target.
- The `K0=20` or `K0=30` runs are much closer to the paper, so the current implementation is better at shrinking toward the target than growing from a strongly under-clustered start.
- Because the backbone features are frozen and imported from upstream SCAN or MoCo checkpoints, the remaining mismatch is most likely concentrated in head initialization, split and merge timing, and threshold dynamics rather than feature extraction itself.


## Table 2

Inferred K convergence compared with the paper.


In [4]:
table2 = table2.sort_values(['Dataset', 'K0']).reset_index(drop=True)
display(table2)


,Dataset,K0,Paper inferred K,Our inferred K,Relative deviation(%)
0,cifar-10,3,10.0,8,20.000000
1,cifar-10,20,10.0,11,10.000000
2,cifar-20,3,19.7,8,59.390863
3,cifar-20,30,19.8,21,6.060606
4,imagenet-10,3,10.0,3,70.000000
5,imagenet-10,20,10.3,11,6.796117
6,stl-10,3,9.7,3,69.072165
7,stl-10,20,10.3,11,6.796117


## Table 3

ImageNet-50 remains separated here as a paper-summary section because the dataset is not available in the current workspace.


In [5]:
display(table3)


,Source,Dataset,Method,Inferred k,ACC(%),NMI(%),ARI(%),Local status
0,paper,imagenet-50,SCAN (k=50),-,73.7±1.7,79.7±0.6,61.8±1.3,NaN
1,paper,imagenet-50,SCAN (k=10),-,19.4±0.1,60.6±0.4,23.0±0.3,NaN
2,paper,imagenet-50,DBSCAN,16.0,24.0±0.0,52.0±0.0,9.0±0.0,NaN
3,paper,imagenet-50,moVB,46.2±1.3,55.0±2.0,70.0±1.0,43.0±1.0,NaN
4,paper,imagenet-50,DPM Sampler,72.0±2.6,57.0±1.0,72.0±0.0,43.0±1.0,NaN
5,paper,imagenet-50,DeepDPM,55.3±1.5,66.0±1.0,77.0±0.0,54.0±1.0,NaN
6,paper,imagenet-50,Ours (paper),50.6±1.7,73.3±1.3,80.1±0.7,62.3±1.5,NaN
7,local,imagenet-50,SCAN + ours,NaN,NaN,NaN,NaN,not_run_missing_imagenet50_dataset
